# Quantum Steganography — My QCC Semester Project

**Course:** Quantum Communication & Cryptography  

---

Okay so the idea came from a question I had during the QKD lecture — if quantum mechanics gives us truly random numbers, why are we still using `random.seed(42)` in every security demo? That bothered me.

I originally planned to just do basic LSB steganography with a quantum key. Then I read about steganalysis and found out that literally any free tool online can detect LSB-hidden data with a chi-square test. So I had to go deeper.

**What I ended up building:**
1. Real quantum key generation — 256 bits from Hadamard circuits (not seeding another RNG)
2. AES-256-GCM encryption of the message *before* hiding it
3. Frequency-domain hiding using DCT (same domain JPEG uses — way harder to detect)
4. Steganalysis tests to prove it actually works
5. A deployable web app (Streamlit)

The main question this project answers: **can you build a steganography system that is both quantum-secured AND statistically invisible?**  
Answer: yes, but it takes more than slapping a Hadamard gate on basic LSB.

## First — why classical steganography tools are actually broken

I want to show this upfront because it's what motivated the whole design.

The standard demo you see everywhere:
- Take an image
- Change the last bit of each pixel to encode your message
- "Look, the image looks the same!"

The problem? That's been broken since 2001. An attacker runs a chi-square test on the pixel histogram and immediately knows you hid something. They don't even need to know *what* — just the fact that something is there defeats the entire purpose of steganography.

I'll demonstrate this at the end when comparing the two approaches.

In [ ]:
# Install everything
# Note: pycryptodome is NOT the same as pycrypto. Don't mix them up — took me a while to figure that out.
!pip install qiskit qiskit-aer pillow scipy pycryptodome numpy matplotlib scikit-image --quiet
print('Done.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.fft import dct, idct
from scipy.stats import chi2
from Crypto.Cipher import AES
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import struct, hashlib, os, json, base64
import warnings
warnings.filterwarnings('ignore')

---
## Part 1 — Quantum Key Generation

The first version of this used the quantum circuit to generate a *seed* for numpy's random number generator. That felt wrong after thinking about it — you're doing classical pseudorandom operations with a fancy-sounding input.

So I changed it: the quantum circuit generates all 256 bits of the AES key directly. 8 qubits, each in superposition via Hadamard, measured once — that's 8 random bits. Repeat 32 times = 256 bits = a full AES-256 key. No classical RNG in the loop.

The quantum bits aren't a shortcut to randomness. They *are* the randomness.

In [ ]:
def generate_quantum_key_256():
    """
    Builds a 256-bit AES key from Hadamard circuits.
    8 qubits x 32 shots = 256 bits. Each measurement is physically random.

    Running on AerSimulator here (accurate simulation).
    For real hardware: swap to qiskit-ibm-runtime — same circuit, different backend.
    """
    simulator = AerSimulator()
    all_bits  = []

    for _ in range(32):
        qc = QuantumCircuit(8, 8)
        for i in range(8):
            qc.h(i)  # superposition — 50/50 on measurement
        qc.measure(range(8), range(8))

        job    = simulator.run(qc, shots=1, memory=True)
        result = job.result()
        bits_this_round = result.get_memory()[0]  # e.g. '10110010'
        all_bits.extend([int(b) for b in bits_this_round])

    # Pack 256 bits into 32 bytes
    key_bytes = bytes(
        int(''.join(map(str, all_bits[i:i+8])), 2)
        for i in range(0, 256, 8)
    )
    return key_bytes, all_bits, key_bytes.hex()


print('Generating quantum key... (takes a few seconds)')
QUANTUM_KEY, KEY_BITS, KEY_HEX = generate_quantum_key_256()

ones  = KEY_BITS.count(1)
zeros = KEY_BITS.count(0)

print(f'\nKey (hex): {KEY_HEX}')
print(f'Length: {len(QUANTUM_KEY)} bytes / {len(KEY_BITS)} bits')
print(f'Distribution: {ones} ones, {zeros} zeros ({100*ones/256:.1f}% / {100*zeros/256:.1f}%)')
print('\nSave this hex string — the receiver needs it to decrypt.')

In [ ]:
# Does the distribution look random?
# A perfectly balanced key = 128 ones and 128 zeros.
# Small deviations are expected and totally fine.

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].bar(['0', '1'], [KEY_BITS.count(0), KEY_BITS.count(1)],
            color=['#4a90d9', '#e84393'], width=0.5)
axes[0].axhline(128, color='gray', linestyle='--', alpha=0.7, label='ideal (50%)')
axes[0].set_title('Quantum Key — Bit Distribution')
axes[0].set_ylabel('Count')
axes[0].legend()

# Visualise key as a 16x16 grid — should look noisy with no visible pattern
key_matrix = np.array(KEY_BITS).reshape(16, 16)
axes[1].imshow(key_matrix, cmap='binary', interpolation='nearest')
axes[1].set_title('Key bits as 16×16 grid (white=1, black=0)')
axes[1].axis('off')

plt.tight_layout()
plt.show()
print('No visible pattern in the grid = good key.')

---
## Part 2 — Encrypting Before Hiding

This was the design decision I'm most convinced about.

Basic steganography hides plaintext. If an attacker detects the stego image (which is easy — more on that later) and extracts the bits, they immediately read your message. So steganography alone is **security through obscurity**. Not real security.

My approach: encrypt first with AES-256-GCM, then hide the ciphertext. Even if someone detects the stego image and extracts every bit, they get encrypted bytes. Without the quantum key, it's useless.

I chose GCM mode because it adds an authentication tag. Wrong key → decryption fails with a MAC error, not garbled output. That matters — garbled text could be mistaken for a partial result. A hard failure is unambiguous.

In [ ]:
def encrypt_message(message: str, key: bytes) -> bytes:
    """
    AES-256-GCM encryption.
    Output layout: nonce (16 bytes) + auth_tag (16 bytes) + ciphertext
    Fixed overhead: 32 bytes per message, regardless of message length.
    """
    cipher = AES.new(key, AES.MODE_GCM)
    ciphertext, tag = cipher.encrypt_and_digest(message.encode('utf-8'))
    return cipher.nonce + tag + ciphertext


def decrypt_message(data: bytes, key: bytes) -> str:
    nonce = data[:16]
    tag   = data[16:32]
    ct    = data[32:]
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    return cipher.decrypt_and_verify(ct, tag).decode('utf-8')


def bytes_to_bits(data: bytes) -> list:
    return [int(b) for byte in data for b in format(byte, '08b')]


def bits_to_bytes(bits: list) -> bytes:
    return bytes(int(''.join(map(str, bits[i:i+8])), 2) for i in range(0, len(bits)-7, 8))


# Quick test
test = 'Quick test of the encryption layer.'
enc  = encrypt_message(test, QUANTUM_KEY)
dec  = decrypt_message(enc, QUANTUM_KEY)

print(f'Original  : {test}')
print(f'Encrypted : {enc.hex()[:64]}... ({len(enc)} bytes)')
print(f'Decrypted : {dec}')
print(f'Match     : {test == dec}')

# Wrong key behaviour
try:
    decrypt_message(enc, os.urandom(32))
except ValueError as e:
    print(f'\nWrong key: "{e}"')
    print('Hard failure, not garbled text — this is correct.')

---
## Part 3 — DCT Steganography (The Core)

This is the part I spent the most time on.

**Why not just LSB?**  
LSB modifies pixel values directly. This creates a specific statistical signature in the pixel histogram — pairs of values (like 200, 201) become artificially equal in frequency. The chi-square test was invented specifically to catch this. It works every time.

**DCT approach:**  
I transform each 8×8 block into the frequency domain using the Discrete Cosine Transform (same as JPEG). Then embed bits into *mid-frequency coefficients* — not DC (overall brightness, too visible) and not the high-frequency corners (lost in compression anyway).

The embedding method is **Quantization Index Modulation (QIM)**: instead of flipping a bit, you nudge a coefficient to the nearest even or odd multiple of a step size (delta). More subtle, more robust.

The *order* of which blocks get used is determined by a hash of the quantum key — so the embedding pattern is also key-dependent.

In [ ]:
BLOCK_SIZE = 8    # Standard 8x8 DCT blocks
DELTA      = 20   # QIM step size. Tested 10, 15, 20, 25 — delta=20 gives PSNR >42dB while staying decodable.

# Mid-frequency positions within an 8x8 DCT block.
# Cross-referenced with the JPEG zigzag scan order.
EMBED_POSITIONS = [(2,1), (1,2), (3,0), (0,3)]  # 4 bits per block


def dct2d(block):  return dct(dct(block.T, norm='ortho').T, norm='ortho')
def idct2d(block): return idct(idct(block.T, norm='ortho').T, norm='ortho')


def embed_bit_qim(coeff: float, bit: int) -> float:
    """
    QIM: round coefficient to nearest DELTA multiple,
    then fix parity to match the bit we want to embed.
    Even q = bit 0. Odd q = bit 1.
    """
    q = int(round(coeff / DELTA))
    if abs(q) % 2 != bit:
        q = q + 1 if q >= 0 else q - 1
    return float(q * DELTA)


def extract_bit_qim(coeff: float) -> int:
    return abs(int(round(coeff / DELTA))) % 2


def get_block_order(total_blocks: int, key: bytes) -> np.ndarray:
    """
    Deterministic block shuffle seeded by the quantum key.
    Same key = same order. Different key = completely different order.
    """
    seed = int.from_bytes(hashlib.sha256(key).digest()[:8], 'big')
    return np.random.default_rng(seed).permutation(total_blocks)


print('DCT engine ready.')
print(f'  Block size      : {BLOCK_SIZE}x{BLOCK_SIZE}')
print(f'  QIM delta       : {DELTA}')
print(f'  Bits per block  : {len(EMBED_POSITIONS)}')

In [ ]:
def encode_image(cover_path: str, message: str, key: bytes, output_path: str) -> dict:
    """
    Full encode pipeline:
      1. Encrypt message
      2. Prepend 4-byte length header (decoder needs to know how much to read)
      3. Convert to bits
      4. Embed into DCT coefficients of quantum-key-ordered blocks
      5. Reconstruct and save
    """
    img   = Image.open(cover_path).convert('RGB')
    arr   = np.array(img, dtype=np.float64)
    h, w, _ = arr.shape
    bh, bw   = h // BLOCK_SIZE, w // BLOCK_SIZE
    total_blocks  = bh * bw
    capacity_bits = total_blocks * len(EMBED_POSITIONS)

    encrypted    = encrypt_message(message, key)
    length_field = struct.pack('>I', len(encrypted))
    payload_bits = bytes_to_bits(length_field + encrypted)

    if len(payload_bits) > capacity_bits:
        raise ValueError(
            f'Image too small for this message.\n'
            f'Need {len(payload_bits)} bits, have {capacity_bits}.\n'
            f'Use a larger image or a shorter message.'
        )

    block_order = get_block_order(total_blocks, key)
    stego       = arr.copy()
    bit_idx     = 0

    for block_num in block_order:
        if bit_idx >= len(payload_bits):
            break
        r0 = (block_num // bw) * BLOCK_SIZE
        c0 = (block_num %  bw) * BLOCK_SIZE

        # Using the Green channel.
        # Human eyes are most sensitive to green luminance,
        # but DCT changes here are well-masked by texture in natural images.
        block     = stego[r0:r0+BLOCK_SIZE, c0:c0+BLOCK_SIZE, 1]
        dct_block = dct2d(block)

        for (r, c) in EMBED_POSITIONS:
            if bit_idx >= len(payload_bits): break
            dct_block[r, c] = embed_bit_qim(dct_block[r, c], payload_bits[bit_idx])
            bit_idx += 1

        stego[r0:r0+BLOCK_SIZE, c0:c0+BLOCK_SIZE, 1] = idct2d(dct_block)

    stego_out = np.clip(stego, 0, 255).astype(np.uint8)
    Image.fromarray(stego_out).save(output_path, 'PNG')

    return {
        'capacity_bits'  : capacity_bits,
        'bits_used'      : len(payload_bits),
        'utilisation_pct': round(100 * len(payload_bits) / capacity_bits, 2),
        'encrypted_bytes': len(encrypted),
        'image_size'     : (h, w)
    }


def decode_image(stego_path: str, key: bytes) -> str:
    """
    Reverse of encode_image.
    Two-pass: read 4-byte header first to know payload length, then read full payload.
    """
    img   = Image.open(stego_path).convert('RGB')
    arr   = np.array(img, dtype=np.float64)
    h, w, _ = arr.shape
    bh, bw   = h // BLOCK_SIZE, w // BLOCK_SIZE
    block_order = get_block_order(bh * bw, key)

    def read_n_bits(n):
        bits = []
        for block_num in block_order:
            if len(bits) >= n: break
            r0 = (block_num // bw) * BLOCK_SIZE
            c0 = (block_num %  bw) * BLOCK_SIZE
            dct_block = dct2d(arr[r0:r0+BLOCK_SIZE, c0:c0+BLOCK_SIZE, 1])
            for (r, c) in EMBED_POSITIONS:
                if len(bits) >= n: break
                bits.append(extract_bit_qim(dct_block[r, c]))
        return bits

    header_bits = read_n_bits(32)
    payload_len = struct.unpack('>I', bits_to_bytes(header_bits))[0]

    if payload_len > 5_000_000:  # sanity check — a 5MB payload would be absurd
        raise ValueError('Payload length unreasonable — probably wrong key or corrupted image.')

    all_bits  = read_n_bits(32 + payload_len * 8)
    enc_bytes = bits_to_bytes(all_bits[32 : 32 + payload_len * 8])
    return decrypt_message(enc_bytes, key)


print('Encode and decode functions ready.')

---
## Part 4 — Running It

For the best results, use a real photograph — richer texture gives the DCT more natural variation to work with. I'm generating a synthetic gradient image here so the notebook runs anywhere without uploading files.

To use your own image: set `COVER_PATH = 'your_image.png'` — everything else stays the same.

In [ ]:
# Synthetic cover image — layered gradients + noise to simulate texture.
# Replace with any real photo for better PSNR results.

np.random.seed(99)
x, y   = np.linspace(0, 3*np.pi, 512), np.linspace(0, 3*np.pi, 512)
xx, yy = np.meshgrid(x, y)

R = np.clip(np.sin(xx) * 100 + np.random.normal(0, 15, (512,512)) + 128, 0, 255).astype(np.uint8)
G = np.clip(np.cos(yy) * 80  + np.random.normal(0, 12, (512,512)) + 128, 0, 255).astype(np.uint8)
B = np.clip(np.sin(xx+yy)*60 + np.random.normal(0, 18, (512,512)) + 110, 0, 255).astype(np.uint8)

cover = Image.fromarray(np.stack([R, G, B], axis=2))
COVER_PATH = 'cover.png'
STEGO_PATH = 'stego.png'
cover.save(COVER_PATH)

plt.figure(figsize=(4, 4))
plt.imshow(cover)
plt.title('Cover image (512×512)')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Image ready. Size: {cover.size}')

In [ ]:
SECRET = (
    "[CONFIDENTIAL] Project codename: QSteg. "
    "This message was AES-256-GCM encrypted using a quantum-generated key "
    "before being embedded into DCT frequency coefficients. "
    "If you can read this, the system worked."
)

print(f'Hiding: "{SECRET[:80]}..."')
print(f'Length: {len(SECRET)} characters\n')

meta = encode_image(COVER_PATH, SECRET, QUANTUM_KEY, STEGO_PATH)

print(f'Done.')
print(f'  Image size       : {meta["image_size"][0]}x{meta["image_size"][1]}')
print(f'  Total capacity   : {meta["capacity_bits"]} bits (~{meta["capacity_bits"]//8 - 36} usable chars)')
print(f'  Used             : {meta["bits_used"]} bits ({meta["utilisation_pct"]}%)')
print(f'  Encrypted size   : {meta["encrypted_bytes"]} bytes')

---
## Part 5 — Does it look the same?

Two metrics:
- **PSNR:** measures overall pixel change. JPEG at 90% quality gives about 38 dB. Above 40 dB is imperceptible.
- **SSIM:** measures structural similarity. 1.0 = identical. Above 0.99 = visually indistinguishable.

In [ ]:
orig  = np.array(Image.open(COVER_PATH).convert('RGB'))
stego = np.array(Image.open(STEGO_PATH).convert('RGB'))

psnr_val = psnr_metric(orig, stego, data_range=255)
ssim_val = ssim_metric(orig, stego, multichannel=True, channel_axis=2, data_range=255)
diff     = np.abs(orig.astype(int) - stego.astype(int))

print(f'PSNR  : {psnr_val:.2f} dB    (JPEG 90% quality reference: ~38 dB)')
print(f'SSIM  : {ssim_val:.6f}')
print(f'Max pixel change  : {diff.max()} out of 255')
print(f'Mean pixel change : {diff.mean():.4f}')
print(f'Pixels changed    : {100*np.sum(diff>0)/orig.size:.2f}%')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig);                                        axes[0].set_title('Original');                      axes[0].axis('off')
axes[1].imshow(stego);                                       axes[1].set_title('Stego image');                   axes[1].axis('off')
axes[2].imshow(np.clip(diff * 25, 0, 255).astype(np.uint8)); axes[2].set_title(f'Diff ×25 | PSNR={psnr_val:.1f}dB'); axes[2].axis('off')
plt.tight_layout()
plt.show()

---
## Part 6 — Steganalysis: Can It Be Detected?

This is the section most steganography projects skip. It's the most important one.

I'm running two attacks:
1. **Chi-square test** — designed to detect LSB steganography by finding statistical imbalances in pixel pair frequencies.
2. **RS analysis** — stronger version that estimates the fraction of image carrying hidden data.

For comparison I'm also creating a basic LSB stego image with the same payload size, running the same attacks on it.

In [ ]:
def chi_square_attack(image: np.ndarray, channel: int = 1) -> float:
    """
    Chi-square steganalysis.
    LSB embedding makes adjacent pixel value pairs (0,1), (2,3)... unnaturally equal.
    Returns p-value — high p = NOT detected.
    """
    pixels = image[:, :, channel].flatten()
    hist   = np.bincount(pixels, minlength=256).astype(float)
    chi_sq = 0.0
    for i in range(0, 255, 2):
        expected = (hist[i] + hist[i+1]) / 2.0
        if expected > 0:
            chi_sq += ((hist[i] - expected)**2 + (hist[i+1] - expected)**2) / expected
    return 1.0 - chi2.cdf(chi_sq, df=127)


def rs_analysis_score(image: np.ndarray, channel: int = 1) -> float:
    """
    RS Analysis. Estimates proportion of image carrying hidden data.
    0.0 = clean. >0.1 = suspicious. Running on 128×128 crop for speed.
    """
    pixels = image[:128, :128, channel].astype(int)
    mask   = [0, 1, 0, 1]

    def f(g): return sum(abs(int(g[i+1]) - int(g[i])) for i in range(len(g)-1))

    R, S, Rn, Sn, total = 0, 0, 0, 0, 0
    for row in range(pixels.shape[0]):
        for col in range(0, pixels.shape[1]-3, 4):
            g = pixels[row, col:col+4]
            if len(g) < 4: continue
            f0 = f(g)

            gf = g.copy()
            for j, m in enumerate(mask):
                if m: gf[j] ^= 1
            f1 = f(gf)

            gn = g.copy()
            for j, m in enumerate(mask):
                if m: gn[j] = gn[j]-1 if gn[j]%2==1 else gn[j]+1
            fn = f(gn)

            if f1 > f0: R += 1
            elif f1 < f0: S += 1
            if fn > f0: Rn += 1
            elif fn < f0: Sn += 1
            total += 1

    if total == 0: return 0.0
    r, s, rn, sn = R/total, S/total, Rn/total, Sn/total
    denom = 2*(r + rn - s - sn)
    return abs((rn - r) / denom) if abs(denom) > 1e-10 else 0.0


# Basic LSB comparison — same payload size
def basic_lsb_embed(cover: np.ndarray, n_bits: int) -> np.ndarray:
    flat = cover.copy().flatten()
    bits = np.random.randint(0, 2, n_bits)
    flat[:n_bits] = (flat[:n_bits] & 0xFE) | bits
    return flat.reshape(cover.shape)

lsb_stego = basic_lsb_embed(orig, meta['bits_used'])

print('Running attacks...\n')

chi_clean = chi_square_attack(orig)
chi_dct   = chi_square_attack(stego)
chi_lsb   = chi_square_attack(lsb_stego)

rs_clean  = rs_analysis_score(orig)
rs_dct    = rs_analysis_score(stego)
rs_lsb    = rs_analysis_score(lsb_stego)

print('Chi-Square (p-value — higher = safer, >0.05 = not detected):')
print(f'  Clean image : {chi_clean:.4f}')
print(f'  DCT system  : {chi_dct:.4f}  → {"PASS" if chi_dct > 0.05 else "DETECTED"}')
print(f'  Basic LSB   : {chi_lsb:.8f}  → {"pass" if chi_lsb > 0.05 else "DETECTED"}')

print()
print('RS Analysis (estimated hidden fraction — lower = safer, <0.1 = clean):')
print(f'  Clean image : {rs_clean:.4f}')
print(f'  DCT system  : {rs_dct:.4f}  → {"PASS" if rs_dct < 0.1 else "suspicious"}')
print(f'  Basic LSB   : {rs_lsb:.4f}  → {"pass" if rs_lsb < 0.1 else "DETECTED"}')

In [ ]:
# The histogram plot that actually shows WHY chi-square catches LSB
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
labels = ['Clean image (baseline)', 'DCT system (ours)', 'Basic LSB steganography']
imgs   = [orig, stego, lsb_stego]
clrs   = ['#4a90d9', '#27ae60', '#e74c3c']

for ax, img, label, color in zip(axes, imgs, labels, clrs):
    hist = np.bincount(img[:,:,1].flatten(), minlength=256)
    ax.bar(range(256), hist, color=color, alpha=0.7, width=1.0)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Pixel value')
    ax.set_ylabel('Count')
    ax.set_xlim(80, 180)  # zoom into mid-range where changes show

plt.suptitle('Green Channel Histogram — Mid-range Zoom\nLSB creates an alternating bar pattern. DCT does not.', fontsize=11)
plt.tight_layout()
plt.show()

---
## Part 7 — Decode and Verify

In [ ]:
print('Decoding...')
recovered = decode_image(STEGO_PATH, QUANTUM_KEY)

print(f'Original : {SECRET}')
print(f'Recovered: {recovered}')
print(f'\nExact match: {SECRET == recovered}')

# Wrong key
print('\n--- Wrong Key Test ---')
try:
    decode_image(STEGO_PATH, os.urandom(32))
except ValueError as e:
    print(f'Result with wrong key: {e}')
    print('Hard error — not garbled output. Correct behaviour.')

In [ ]:
# Export key
key_package = {
    'key_hex'         : KEY_HEX,
    'key_b64'         : base64.b64encode(QUANTUM_KEY).decode(),
    'algorithm'       : 'AES-256-GCM + DCT-QIM steganography',
    'quantum_circuit' : '8-qubit Hadamard, 32 measurement rounds (AerSimulator)',
    'note'            : 'Exchange this key via a separate secure channel. Never send it with the stego image.'
}

with open('key_package.json', 'w') as f:
    json.dump(key_package, f, indent=2)

print('Key exported to key_package.json')
print(f'\nKey (hex): {KEY_HEX}')
print(f'Key (b64): {key_package["key_b64"]}')

---
## Results

In [ ]:
print('=' * 60)
print('RESULTS')
print('=' * 60)
print()
print('Quantum Key')
print(f'  8-qubit Hadamard × 32 rounds = {len(QUANTUM_KEY)*8} bits')
print(f'  Bit balance: {100*KEY_BITS.count(1)/256:.1f}% ones / {100*KEY_BITS.count(0)/256:.1f}% zeros')
print()
print('Encryption')
print(f'  AES-256-GCM | payload: {meta["encrypted_bytes"]} bytes | auth: 128-bit tag')
print()
print('Steganography')
print(f'  Frequency domain DCT, 8×8 blocks, QIM delta={DELTA}')
print(f'  Block order: quantum-key-seeded permutation')
print(f'  Used {meta["bits_used"]} / {meta["capacity_bits"]} bits ({meta["utilisation_pct"]}%)')
print()
print('Image Quality')
print(f'  PSNR : {psnr_val:.2f} dB  (JPEG 90% ref: ~38 dB)')
print(f'  SSIM : {ssim_val:.6f}')
print()
print('Steganalysis')
print(f'  Chi-square : p={chi_dct:.4f}  → {"PASS" if chi_dct > 0.05 else "FAIL"}')
print(f'  RS analysis: {rs_dct:.4f}    → {"PASS" if rs_dct < 0.1 else "FAIL"}')
print('=' * 60)

---
## Limitations — things I'm being upfront about

**AerSimulator is not real quantum hardware.**  
The simulator accurately models quantum mechanics, but the underlying randomness source is a classical PRNG inside the simulator. For actual quantum randomness you'd need IBM Quantum hardware via `qiskit-ibm-runtime`. The circuit and key generation code are identical — only the backend changes.

**JPEG will destroy the hidden data.**  
JPEG's lossy compression modifies DCT coefficients and breaks the embedded bits. Always save stego images as PNG.

**Key distribution is still a real problem.**  
The quantum key needs to reach the receiver securely. I export it manually in this project. In a real system you'd use QKD or asymmetric key exchange. This is not specific to this project — it's the fundamental problem in all symmetric-key systems.

**Modern deep-learning steganalysis is out of scope.**  
This system resists chi-square and RS analysis. It has not been tested against ML-based detectors like SRNet or Zhu-Net, which are trained specifically on frequency-domain steganography. Defeating those requires adaptive content-aware embedding — an active research problem.

**Capacity scales with image size.**  
A 256×256 image can hold roughly 500 characters. A 1024×1024 can hold ~8000.